# Init
---

In [1]:
#### mv packages ####
import modules.data as d
import modules.model as m
import modules.pooling as p
import modules.train as t
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device, generator = u.Devices().auto_set_device(['cuda:1', 'cuda:0'])

#### data ####
brca = d.Preprocessor(
    tcga_project='TCGA-BRCA',
    tcga_dir=dataset_dir/'tcga',
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',
    
    # counts
    apply_DESeq_norm=True, 
    log_transform=True,
    scale_method='standard',

    # etc
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Primary Tumor']},
    max_subset = 120,
)
_dataset = d.GraphDataset(brca)
_batch = d.get_toy_databatch(_dataset, generator)

# #### Device() ####
# device = cuda:4

# #### Preprocessor() ####
# log0_method              log1p                    str
# class_weights            (6,)                     Tensor (cuda:4)
# edge_index               (2, 32798)               Tensor (cuda:4)
# edge_attr                (32798, 16)              Tensor (cuda:4)
# gene_counts              (4383, 562)              DataFrame
# metadata                 (562, 3)                 DataFrame
# relation                 (32798, 18)              DataFrame
# node_id_map              4383                     dict
# mask_list                305                      list
# mask                     (4383, 305)              Tensor (cuda:4)
# x                        (562, 4383, 1)           Tensor (cuda:4)
# y                        (562,)                   Tensor (cuda:4)
# y_labels                 6                        list
# num_samples              562                      int
# num_nodes                4383                     int


# Encoder
---

In [2]:
import torch
import torch.nn as nn

from modules.model import SequentialModel
from torch import Tensor
from torch_geometric.nn import MessagePassing
from typing import Union

In [3]:
class Node2SetEncoder(nn.Module):
    def __init__(
        self, 
        num_node_features:int, # in_channels
        embedding_size:int, # out_channels
        mask:Tensor, # (num_nodes, num_sets)
        pooling_class:p.SetPooling,
        encoder_class:Union[nn.Module, MessagePassing],
        pooling_kwargs:dict={},  
        encoder_kwargs:dict={},
        debug:bool=False, 
        *args, 
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.debug = debug
        self.num_nodes, self.num_sets = mask.shape

        self.node_encoder = SequentialModel(
            in_channels=num_node_features,
            out_channels=embedding_size,
            layer_class=encoder_class,
            **encoder_kwargs
        )

        self.node2set = pooling_class(
            out_channels=embedding_size,
            mask=mask,
            **pooling_kwargs
        )

        self.set2sample = pooling_class(
            out_channels=embedding_size,
            mask=torch.ones(self.num_sets,1),
            **pooling_kwargs
        )

    def forward(self, x:Tensor, return_set_embedding:bool=True, *args, **kwargs):
        # x: (batch_size * num_nodes, embedding_size), from PyG DataBatch
        batch_size = int(x.shape[0] / self.num_nodes)

        # get node embedding
        node_emb = self.node_encoder(x, *args, **kwargs) # (batch_size * num_nodes, embedding_size)
        node_emb = node_emb.view(batch_size, self.num_nodes, -1) # (batch_size, num_nodes, embedding_size)

        # node to set embedding
        set_emb = self.node2set(node_emb) # (batch_size, num_sets, embedding_size)

        # set to sample embedding
        samp_emb = self.set2sample(set_emb) # (batch_size, 1, embedding_size)
        samp_emb = samp_emb.squeeze(1) # (batch_size, embedding_size)

        if self.debug:
            return {'node_emb': node_emb, 'set_emb': set_emb, 'samp_emb': samp_emb}
        return {'samp_emb': samp_emb, 'set_emb': set_emb}

In [17]:
# predefined
_embedding_size = 128

# from mask (init)
_mask = brca.mask
_num_nodes, _num_sets = _mask.shape

# from batch(forward)
_batch_size = int(_batch.x.shape[0]/_num_nodes)
_num_node_features = _batch.x.shape[1] # or brca.num_node_features

In [26]:
_encoder = Node2SetEncoder(
    num_node_features=brca.num_node_features,
    embedding_size=_embedding_size,
    mask=brca.mask,
    pooling_class=p.SoftmaxSP,
    encoder_class=nn.Linear
)

out = _encoder(_batch.x, edge_index=_batch.edge_index)
_samp_emb = out['samp_emb']

# Simple Decoder Autoencoder
---
Benchmark, make sure encoder works


In [34]:
_mlp_decoder = SequentialModel(
    in_channels=_embedding_size,
    out_channels=brca.num_nodes,
    layer_class=nn.Linear
)

_mlp_decoder(_samp_emb).view(-1, 1).view(-1).shape

torch.Size([280512])

In [ ]:
class SimpleDecoder(nn.Module):
    def __init__(
        self,
        num_node_features:int,
        embedding_size:int,
        mask:Tensor, # not used

        # classes
        pooling_class:nn.Module=None, # not used
        decoder_class:nn.Module=nn.Linear,

        # kwargs
        pooling_kwargs:dict={}, # not used
        decoder_kwargs:dict={},

        # etc.
        debug:bool=False,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.decoder = SequentialModel(
            in_channels=embedding_size,
            out_channels=num_node_features,
            layer_class=decoder_class,
            **decoder_kwargs
        )
    
    def extract_encoder_out(self, encoder_out:dict):
        # get encoder outputs; return
        node_emb = encoder_out.get('node_emb')
        set_emb = encoder_out.get('set_emb')
        samp_emb = encoder_out.get('samp_emb')
        return node_emb, set_emb, samp_emb
        
    def forward(self, encoder_out, *args, **kwargs):
        # extract encoder outputs
        node_emb, set_emb, samp_emb = self.extract_encoder_out(encoder_out)
        node_recon = self.decoder(samp_emb)
        return node_recon
        
        



In [ ]:
class Autoencoder(nn.Module):
    def __init__(
        self,
        num_node_features:int,
        embedding_size:int,
        mask:Tensor,

        # classes
        pooling_class:p.SetPooling,
        encoder_class:Union[nn.Module,MessagePassing],
        decoder_class:nn.Module,

        # kwargs
        pooling_kwargs:dict={},
        encoder_kwargs:dict={},
        decoder_kwargs:dict={},

        # etc.
        debug:bool=True,
        *args, 
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.encoder = encoder_class(
            num_node_features=num_node_features,

        )